# COMP8851 — T-Finance and T-Social

Completes the two datasets missing from the Vast.ai run, for **CARE-GNN** and
**GHRN**.

## Two settings, then Run All

1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. Upload `COMP8851-vastai-FIXED.zip` as a Kaggle Dataset, add it with
   *+ Add Input*, and set `REPO_ZIP` below if the path differs.

Then **Run All**. Everything else — downloading the graphs, decoding them,
training, archiving — is automatic. Nothing needs watching.

## How this is made fast

The expensive part of this benchmark is the hyperparameter search, not the
training. So the search is cut hard and the training budget is left alone:

| | Full protocol | Here |
|---|---|---|
| Tuning trials | 12 | **2** |
| Tuning epochs | 100 | **10** |
| Final epochs | 100 | **100** (unchanged) |
| Patience | 20 | **20** (unchanged) |
| Seeds | 2, 42, 72 | 2, 42, 72 (unchanged) |

Two trials at 10 epochs instead of twelve at 100 removes roughly 98% of the
search cost. The final runs keep the full budget, which is what makes these
cells comparable with the eight already completed — and early stopping means
they rarely reach 100 anyway (the completed runs stopped at 46-86 epochs).

Shrinking the *graph* would be a different matter: a subsampled T-Social is not
T-Social, and the result would not be a benchmark number for that dataset. The
graphs are used whole.

## What will fit on a T4

T-Social used **~34 GB of GPU memory** on the A6000. A Kaggle T4 has **16 GB**,
and Kaggle gives ~30 GB of system RAM against the A6000 host's 755 GB.

| Cell | Expectation |
|---|---|
| CARE-GNN × T-Finance | should complete |
| GHRN × T-Finance | should complete |
| GHRN × T-Social | may exhaust memory |
| CARE-GNN × T-Social | very unlikely to fit |

T-Finance results are archived **before** T-Social is attempted, so a crash
there cannot cost you the cells that worked. An out-of-memory failure is
recorded with its traceback and reports as **FAILED_TECHNICAL** — one of the
five statuses the protocol defines, and better evidence than an unattempted
cell.

## Hardware

These runs are on a **Kaggle Tesla T4**, not the A6000 behind the other eight
cells. Protocol v4.4 forbids pooling timings across hardware classes. AUROC,
AUPRC and F1 stay directly comparable; timings do not. `build_deliverable.py`
reads each run's `hardware.json` and splits the efficiency tables by GPU
automatically.


In [ ]:
# ---------------------------------------------------------------- settings
REPO_ZIP   = "/kaggle/input/comp8851-repo/COMP8851-vastai-FIXED.zip"  # <-- edit if needed
DO_TSOCIAL = True     # set False to stop after T-Finance

TRIALS, TUNE_EPOCHS, TUNE_PATIENCE = 2, 10, 5
FINAL_EPOCHS, FINAL_PATIENCE = 100, 20      # protocol budget; leave these alone
SEEDS = ["2", "42", "72"]
# ---------------------------------------------------------------------------

import os, sys, time, json, shutil, zipfile, subprocess, traceback
from pathlib import Path

WORK = Path("/kaggle/working/comp8851")
OUTDIR = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")

STATUS = {}          # every step records its outcome here

def step(name):
    """Run a step, never let it kill the notebook, always record what happened."""
    def wrap(fn):
        print(f"\n{'='*70}\n{name}\n{'='*70}")
        started = time.time()
        try:
            result = fn()
            STATUS[name] = {"ok": True, "minutes": (time.time()-started)/60}
            return result
        except Exception as exc:
            STATUS[name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}",
                            "minutes": (time.time()-started)/60}
            print(f"FAILED: {type(exc).__name__}: {exc}")
            traceback.print_exc(limit=3)
            return None
    return wrap

import torch
GPU_GB = 0.0
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_GB = p.total_memory / 1024**3
    print(f"GPU   : {p.name}  ({GPU_GB:.1f} GB)")
else:
    print("NO GPU DETECTED -- enable it in Settings, then Run All again.")
print(f"torch : {torch.__version__}")

if not Path(REPO_ZIP).exists():
    cands = list(Path("/kaggle/input").rglob("*.zip"))
    print(f"\nRepo zip not at {REPO_ZIP}")
    print("Zips found under /kaggle/input:", [str(c) for c in cands][:5])
    if len(cands) == 1:
        REPO_ZIP = str(cands[0])
        print(f"Using {REPO_ZIP}")
    else:
        raise SystemExit("Set REPO_ZIP to the correct path and Run All again.")

with zipfile.ZipFile(REPO_ZIP) as z:
    z.extractall("/kaggle/working/_repo")
inner = next(Path("/kaggle/working/_repo").rglob("scripts/run_benchmark.py")).parents[1]
for item in inner.iterdir():
    target = WORK / item.name
    if not target.exists():
        shutil.copytree(item, target) if item.is_dir() else shutil.copy2(item, target)
os.chdir(WORK)
print(f"\nrepo  : {WORK}")

## 1. Dependencies

Kaggle's PyTorch has no matching DGL wheel, and T-Finance/T-Social ship **as
DGL binary graphs**. The repo handles this with an isolated decoder
environment: a separate venv with an older torch plus a matching DGL, used only
to read those two files and write the canonical `.npz`. Training then runs in
this notebook's own environment from the `.npz`, with GHRN on its
`torch.sparse` backend.

The decoded graph is identical either way — the `.npz` round-trip is verified
byte-for-byte including the view hash.

In [ ]:
@step("1. dependencies")
def _():
    pkgs = ["gdown", "scikit-learn", "pandas", "scipy", "matplotlib", "PyYAML", "psutil"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

    def dgl_ok():
        r = subprocess.run([sys.executable, "-c", "import dgl; print(dgl.__version__)"],
                           capture_output=True, text=True)
        return (r.stdout.strip() if r.returncode == 0 else None)

    version = dgl_ok()
    if version:
        print("DGL already importable:", version)
        return True

    # Fast path: if this torch has a matching DGL wheel, installing it directly
    # is minutes rather than the ~3 GB isolated environment the decoder builds.
    major_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_tag = "cu121" if torch.version.cuda and torch.version.cuda.startswith("12.1") else None
    indexes = [f"https://data.dgl.ai/wheels/torch-{major_minor}/repo.html"]
    if cuda_tag:
        indexes.insert(0, f"https://data.dgl.ai/wheels/torch-{major_minor}/{cuda_tag}/repo.html")

    for index in indexes:
        print(f"  trying DGL from {index}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dgl", "-f", index],
                       capture_output=True, text=True)
        version = dgl_ok()
        if version:
            print(f"  DGL installed directly: {version} -- decoding will be fast")
            return True

    print("  no DGL wheel for this torch. The isolated decoder will handle "
          "T-Finance / T-Social automatically during the freeze step.")
    print("  (That builds a separate ~3 GB environment and takes a few minutes.)")
    return True

## 2. Download the graphs

From the BWGNN authors' release. T-Social is ~744 MB compressed, ~4.1 GB
extracted, so this takes a few minutes. The archives are deleted afterwards to
reclaim disk.

In [ ]:
@step("2. download")
def _():
    import gdown
    FOLDER = "https://drive.google.com/drive/folders/1PpNwvZx_YRSCDiHaBUmRIS3x1rZR7fMr"
    (WORK / "data").mkdir(exist_ok=True)
    gdown.download_folder(FOLDER, output=str(WORK / "data/bwgnn_drive"),
                          quiet=False, use_cookies=False)

    wanted = ["tfinance", "tsocial"] if DO_TSOCIAL else ["tfinance"]
    for name in wanted:
        src = WORK / f"data/bwgnn_drive/dataset/{name}.zip"
        dst = WORK / "data" / name
        dst.mkdir(parents=True, exist_ok=True)
        if not src.exists():
            print(f"{name}: archive not downloaded")
            continue
        subprocess.run(["unzip", "-q", "-o", str(src), "-d", str(dst)], check=False)
        # the loader wants the graph file itself at data/<name>/<name>
        found = [p for p in dst.rglob(name) if p.is_file()]
        if found and found[0] != dst / name:
            shutil.move(str(found[0]), str(dst / name))
        target = dst / name
        print(f"{name}: {target.stat().st_size/1024**2:.0f} MB"
              if target.exists() else f"{name}: NOT FOUND after unzip")

    for z in (WORK / "data/bwgnn_drive").rglob("*.zip"):
        z.unlink()
    subprocess.run(["df", "-h", "/kaggle/working"])
    return True

## 3. Decode to canonical format

Each dataset is frozen separately so a failure on one cannot cost the other.
T-Social is the memory-hungry one.

In [ ]:
def freeze(name):
    r = subprocess.run([sys.executable, "shared/comp8851/fetch_datasets.py",
                        "--data-root", "data", "--freeze", "--only", name],
                       capture_output=True, text=True, cwd=str(WORK))
    for line in (r.stdout or "").strip().splitlines()[-12:]:
        print("  ", line)
    if r.returncode != 0 and r.stderr:
        print("   STDERR:", r.stderr[-500:])
    cache = list((WORK / "data" / name).glob("*_canonical.npz"))
    print(f"   -> {'OK ' + cache[0].name if cache else 'FAILED to freeze ' + name}")
    return bool(cache)

tfinance_ok = step("3a. freeze T-Finance")(lambda: freeze("tfinance")) or False
tsocial_ok = (step("3b. freeze T-Social")(lambda: freeze("tsocial")) or False) if DO_TSOCIAL else False

## 4. Train

Streams progress so you can see it working. Each pair is independent.

In [ ]:
def run_pair(model, dataset, trials, tune_epochs, minutes, extra=None):
    if not (WORK / "data" / dataset).exists():
        print(f"SKIP {model} x {dataset}: dataset absent")
        return False
    cmd = [sys.executable, "scripts/run_benchmark.py",
           "--models", model, "--datasets", dataset,
           "--ratios", "TR40", "--seeds", *SEEDS,
           "--trials", str(trials),
           "--tune-epochs", str(tune_epochs), "--tune-patience", str(TUNE_PATIENCE),
           "--epochs", str(FINAL_EPOCHS), "--patience", str(FINAL_PATIENCE),
           "--max-minutes", str(minutes),
           "--data-root", "data", "--results-root", "results",
           "--no-archive", "--no-report"] + (extra or [])
    proc = subprocess.Popen(cmd, cwd=str(WORK), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    keep = ("trial", "COMPLETE", "FAILED", "seed", "isolation", "epoch 1",
            "Error", "error", "Traceback", "memory")
    for line in proc.stdout:
        if any(k in line for k in keep):
            print("  ", line.rstrip())
    proc.wait()
    print(f"   exit {proc.returncode}")
    return proc.returncode == 0

if tfinance_ok:
    step("4a. CARE-GNN x T-Finance")(
        lambda: run_pair("CARE-GNN", "tfinance", TRIALS, TUNE_EPOCHS, 45))
    step("4b. GHRN x T-Finance")(
        lambda: run_pair("GHRN", "tfinance", TRIALS, TUNE_EPOCHS, 45))
else:
    print("T-Finance unavailable; skipping both T-Finance cells.")

## 5. Archive T-Finance immediately

Before T-Social is attempted. If that run takes the kernel down, this file
already exists.

In [ ]:
@step("5. archive T-Finance")
def _():
    if not (WORK / "results").exists():
        print("no results yet")
        return False
    shutil.make_archive(str(OUTDIR / "tfinance_results"), "zip", str(WORK / "results"))
    mb = (OUTDIR / "tfinance_results.zip").stat().st_size / 1024**2
    print(f"saved tfinance_results.zip ({mb:.1f} MB) - download it from the Output panel")
    return True

## 6. T-Social

Attempted rather than assumed. GHRN first — it has the better chance, because
its beta-wavelet backbone works on sparse tensors instead of materialising
per-node adjacency structures. CARE-GNN needs `--allow-large` to pass its node
guard and is unlikely to survive Kaggle's RAM, but the protocol says attempt
every pair and record the outcome.

A failure here **is a result**. Do not delete it.

In [ ]:
if tsocial_ok:
    print(f"GPU has {GPU_GB:.1f} GB; T-Social used ~34 GB on the A6000.")
    if GPU_GB < 30:
        print("Below what it needed there. Attempting anyway so the outcome is "
              "measured rather than assumed.")
    step("6a. GHRN x T-Social")(
        lambda: run_pair("GHRN", "tsocial", 2, TUNE_EPOCHS, 180))
    step("6b. CARE-GNN x T-Social")(
        lambda: run_pair("CARE-GNN", "tsocial", 2, TUNE_EPOCHS, 180,
                         extra=["--allow-large"]))
elif DO_TSOCIAL:
    print("T-Social did not freeze - most likely RAM or disk during the freeze "
          "step. That is a recordable blocked outcome, not a silent gap.")
else:
    print("T-Social skipped by request (DO_TSOCIAL = False).")

## 7. Package and report

In [ ]:
@step("7. package")
def _():
    r = subprocess.run([sys.executable, "scripts/build_deliverable.py",
                        "--results", "results",
                        "--out", str(OUTDIR / "DELIVERABLE_KAGGLE"),
                        "--platform", "Kaggle (Tesla T4)"],
                       capture_output=True, text=True, cwd=str(WORK))
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1000:])
    shutil.make_archive(str(OUTDIR / "tfinance_tsocial_results"), "zip",
                        str(WORK / "results"))
    return True

print(f"\n{'='*70}\nRUN SUMMARY\n{'='*70}")
for name, info in STATUS.items():
    mark = "OK  " if info["ok"] else "FAIL"
    detail = "" if info["ok"] else f"  {info['error'][:70]}"
    print(f"  {mark}  {name:<32} {info['minutes']:5.1f} min{detail}")

print(f"\n{'='*70}\nFILES TO DOWNLOAD (Output panel, right)\n{'='*70}")
for f in sorted(OUTDIR.glob("*.zip")):
    print(f"  {f.name:<38} {f.stat().st_size/1024**2:8.1f} MB")

cells_done = len(list((WORK / "results").rglob("test_metrics.json"))) if (WORK/"results").exists() else 0
print(f"\ncompleted test evaluations: {cells_done}")
print("\nSend the zips on to have them merged with the A6000 results into one "
      "deliverable, with the two hardware classes reported separately.")